# TaoConfig: Capturing and Restoring Tao State

`TaoConfig` captures the complete configuration of a running Tao session, including Bmad common settings, beam parameters, global settings, and per-element overrides.

This makes it possible to:
- Save the exact state of a Tao session
- Restore it later (even on a different machine)
- Generate standalone shell scripts that reproduce the session
- Diff configurations to see what changed

## Setup

In [ ]:
from pytao import Tao

tao = Tao(init_file="$ACC_ROOT_DIR/bmad-doc/tao_examples/cesr/tao.init", plot="mpl")

## Getting the current configuration

Use `tao.get_config()` to snapshot the current Tao state.

In [ ]:
config = tao.get_config()
config

### Inspecting sub-models

A `TaoConfig` contains several sub-models, each corresponding to a Tao settings group.

In [ ]:
# Startup parameters (lattice file, init file, etc.)
config.startup

In [ ]:
# Bmad common settings
config.com

In [ ]:
# Global Tao settings
config.globals

In [ ]:
# Beam initialization parameters
config.beam_init

In [ ]:
# Beam tracking parameters
config.beam

## Modifying and applying configurations

You can modify a config and apply it back to Tao. This generates and runs the appropriate `set` commands.

In [ ]:
# Modify a setting
original_value = config.beam_init.a_emit
config.beam_init.a_emit = 2e-8

# See what set commands would be generated
print("All set commands for beam_init:")
for cmd in config.beam_init.set_commands[:5]:
    print(f"  {cmd}")
print(f"  ... ({len(config.beam_init.set_commands)} total)")

In [ ]:
# We can also see just the ones that have changed:

print("All set commands for beam_init that differ from the original config:")
for cmd in config.beam_init.get_set_commands(tao):
    print(f"  {cmd}")

In [ ]:
# Apply configuration to Tao
config.set(tao)

In [ ]:
# Restore original value
config.beam_init.a_emit = original_value
config.set(tao)

## Per-element settings

You can store element-specific settings in the configuration too.

In [ ]:
# Add per-element overrides
config.settings_by_element = {
    "Q00W": {"k1": "0.5"},
    "Q01W": {"k1": "0.3"},
}

# See the generated commands
for cmd in config.per_element_commands:
    print(cmd)

## Saving and loading configurations

`TaoConfig` supports JSON, compressed JSON, and HDF5 serialization.

In [ ]:
from pathlib import Path

config = tao.get_config()

# Save to JSON
config.write("my_config.json")

# Save to compressed JSON
config.write("my_config.json.gz");

In [ ]:
from pytao.model import TaoConfig

# Load from file
loaded_config = TaoConfig.from_file(Path("my_config.json"))
print("Loaded startup - init_file:", loaded_config.startup.init_file)

In [ ]:
loaded_config == config

## Archiving a Tao session

The `tao.archive()` method combines storing configuration details and a lattice snapshot into a specific directory.

It writes:
1. The current lattice (via `write bmad`)
2. All configuration settings as a Tao command file
3. A bash script that launches Tao with the lattice and applies the settings

This is useful for reproducibility: you can recreate the exact Tao session on any machine with Tao installed.

In [ ]:
sh_file, cmd_file = tao.archive("my_archive")
print(f"Shell script: {sh_file}")
print(f"Command file: {cmd_file}")

In [ ]:
# View the generated shell script
print(sh_file.read_text())

In [ ]:
# View a snippet of the command file
cmd_lines = cmd_file.read_text().splitlines()
print(f"Total lines: {len(cmd_lines)}")
print("First 10 commands:")
for line in cmd_lines[:10]:
    print(f"  {line}")

In [ ]:
# Cleanup
import os, shutil
for fn in ["my_config.json", "my_config.json.gz"]:
    if os.path.exists(fn):
        os.remove(fn)
if os.path.exists("my_archive"):
    shutil.rmtree("my_archive")